# Environment

In [1]:
import os
os.chdir("..")

In [2]:
import pandas as pd
import spacy

In [3]:
nlp = spacy.load("pt_core_news_lg")

### Load Lexicons

In [4]:
liwc = pd.read_csv("./data/lexicons/liwc.csv")
sentilex = pd.read_csv("./data/lexicons/sentilex.csv")
wordnetaffect = pd.read_csv("./data/lexicons/wordnetaffect.csv")
lex = pd.concat([liwc, sentilex, wordnetaffect]).sort_values(by='polarity').drop_duplicates(subset=['word'], keep='first').reset_index(drop=True)

### Filter Polarities

In [5]:
pos_set = set(lex.loc[lex.polarity == 1, 'word'].str.lower())
neg_set = set(lex.loc[lex.polarity == -1, 'word'].str.lower())
nto_set = set(lex.loc[lex.polarity == 0, 'word'].str.lower())

### Lexical-Based Features Extraction

In [6]:
def extract_lex_features(text):
    doc = nlp(text)
    toks_lower = [token.text.lower() for token in doc if not token.is_stop]
    toks_lemma_lower = [token.lemma_.lower() for token in doc if not token.is_stop]
    n = max(len(toks_lower), 1)

    pos_tokens = [t for t in toks_lower if t in pos_set] + [t for t in toks_lemma_lower if t in pos_set]
    pos_count = len(pos_tokens)

    neg_tokens = [t for t in toks_lower if t in neg_set] + [t for t in toks_lemma_lower if t in neg_set]
    neg_count = len(neg_tokens)

    nto_tokens = [t for t in toks_lower if t in nto_set] + [t for t in toks_lemma_lower if t in nto_set]
    nto_count = len(nto_tokens)
    return {
        "feat_pos_ratio": round(pos_count/n, 2),
        "feat_neg_ratio": round(neg_count/n, 2),
        "feat_nto_ratio": round(nto_count/n, 2),
    }

### Extraction of AMIVE

In [7]:
df = pd.read_csv("./data/processed/amive.csv")

In [8]:
feat_list = df['TEXT'].apply(extract_lex_features).to_list()

In [9]:
df_lex = df[["DOCNO"]].join(pd.DataFrame(feat_list, index=df.index))

In [10]:
df_lex

,DOCNO,feat_pos_ratio,feat_neg_ratio,feat_nto_ratio
0,100_10,0.00,0.20,0.90
1,100_11,0.00,0.67,0.33
2,100_12,0.00,0.40,1.20
3,100_13,0.00,0.50,0.75
4,100_14,0.00,0.00,0.00
...,...,...,...,...
1535,99_12,0.00,0.00,0.00
1536,99_15,0.07,0.29,0.43
1537,99_16,0.17,0.42,0.71
1538,99_2,0.00,0.54,0.54


In [11]:
df_lex.to_csv("./data/features/amive_lex.csv", index=False)